# Sections 1 and 2: Positioning and Direct Replication

ECON 5200 Midterm Project · Replication Study

> **🛑 AI policy, and it is not the problem sets' policy.** Sections 1 and 2 — the
> positioning and the direct replication — are **no AI**. Not for the code, not for
> the prose. Making a computer agree with a published table is the exercise that
> establishes you can read a paper, and a model that has already read the paper will
> simply hand you the answer without your ever finding the discrepancy. Sections 3 to
> 6 authorise AI as a documented co-pilot under **P.R.I.M.E.**, on the same terms as
> every problem set: you file the full transcript in `ai-appendix.md`, and if you
> cannot explain a line you cannot submit it. State in your README which sections you
> used it in. This follows the course rule — manual command first, expansion second —
> and using AI in Sections 1 or 2 is an integrity violation, not a style disagreement.

## Setup

Imports and paths only.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

## Data

In [2]:
# Set MY_PAPER once, in download_data.py. fetch() downloads into data/ only if a file is missing.
if MY_PAPER:
    paths = fetch(MY_PAPER)
    print(PAPERS[MY_PAPER]["paper"])
    # TODO: read the file(s), e.g. df = pd.read_stata(paths[0])
else:
    print("TODO: set MY_PAPER in download_data.py (castle, card, lee, dw, hansen or abortion).")

  data/lmb-data.dta  7.2 MB
Lee (2008), incumbency advantage in US House elections


In [4]:
import pandas as pd

df = pd.read_stata("../data/lmb-data.dta")

print(df.shape)
print(df.columns.tolist())
df.head()

(13588, 178)
['state', 'district', 'incmbncy', 'demvote', 'repvote', 'year', 'congress', 'occupanc', 'name', 'eq_Dwhip', 'eq_Rwhip', 'eq_Dlead', 'eq_Rlead', 'vote', 'republic', 'party', 'office', 'icpsr_id', 'whip_D', 'whip_R', 'vote_tot', 'demvoteshare', 'dembin', 'lagdemvoteshare', 'clusterid', 'pooleyear', 'redistrict', 'poolename', 'aclu_vs', 'acu_vs', 'aca_vs', 'ada_vs', 'afbf_vs', 'afge_vs', 'afscme_vs', 'aft_vs', 'asc_vs', 'bfw_vs', 'bctd_vs', 'carter_vs', 'ccus1_vs', 'ccus2_vs', 'cwla_vs', 'cv_vs', 'cvvf_vs', 'sane_vs', 'cfnfmp_vs', 'aflcio_vs', 'cfsca_vs', 'cfsce_vs', 'cfscd_vs', 'cfscs_vs', 'ccause_vs', 'cw_vs', 'cc_vs', 'cfa_vs', 'ike_vs', 'pfpike_vs', 'pdpike_vs', 'ford_vs', 'fcnl_vs', 'lbj_vs', 'pfplbj_vs', 'pdplbj_vs', 'jfk_vs', 'pfpjfk_vs', 'pdpjfk_vs', 'lfr_vs', 'lcv_vs', 'lwv_vs', 'll_vs', 'lfs_vs', 'nasc_vs', 'ncsc_vs', 'nea_vs', 'nfo_vs', 'nfu_vs', 'nfib_vs', 'ntu_vs', 'nwpc_vs', 'nr_vs', 'nixon_vs', 'reagan_vs', 'ripon_vs', 'twr_vs', 'uaw_vs', 'umw_vs', 'firstyear',

,state,district,incmbncy,demvote,repvote,year,congress,occupanc,name,eq_Dwhip,...,pres_rep_pct,n,lagged,_merge,democrat,lagdemocrat,score,lagscore,id,id2
0,01: Connecticut,1,-1: REPUBLICAN,127802.0,103294.0,1948.0,81.0,0.0,"RIBICOFF,",0.752336,...,NaN,1.0,14.760000,3,1.0,0.0,64.339996,14.760000,3.0,1.0
1,01: Connecticut,1,-1: REPUBLICAN,127802.0,103294.0,1948.0,81.0,0.0,"RIBICOFF,",0.752336,...,NaN,2.0,9.520000,3,1.0,0.0,60.279999,9.520000,4.0,1.0
2,01: Connecticut,1,1: DEMOCRAT,134258.0,96251.0,1950.0,82.0,0.0,"RIBICOFF,",0.770270,...,NaN,1.0,64.339996,3,1.0,1.0,57.060001,64.339996,5.0,1.0
3,01: Connecticut,1,1: DEMOCRAT,134258.0,96251.0,1950.0,82.0,0.0,"RIBICOFF,",0.770270,...,NaN,2.0,60.279999,3,1.0,1.0,73.830002,60.279999,6.0,1.0
4,01: Connecticut,1,1: DEMOCRAT,148935.0,112526.0,1954.0,84.0,0.0,DODD,0.730769,...,NaN,1.0,56.220001,3,1.0,1.0,42.959999,56.220001,7.0,1.0


In [5]:
keywords = ["vote", "margin", "share", "win", "dem", "rep"]

for col in df.columns:
    if any(word in col.lower() for word in keywords):
        print(col)

demvote
repvote
vote
republic
vote_tot
demvoteshare
dembin
lagdemvoteshare
demvs2
demvs3
demvs4
lagdemvs2
lagdemvs3
lagdemvs4
dem2
lagdem2
pres_dem_pct
pres_rep_pct
democrat
lagdemocrat


In [6]:
cols = [
    "year",
    "demvoteshare",
    "lagdemvoteshare",
    "dembin",
    "democrat",
    "lagdemocrat"
]

print(df[cols].describe().round(3))
print(df[cols].head(10))

            year  demvoteshare  lagdemvoteshare     dembin   democrat  \
count  13588.000     13577.000        13577.000  13577.000  13588.000   
mean    1969.459         0.582            0.577      0.578      0.597   
std       12.754         0.230            0.226      0.230      0.491   
min     1948.000         0.000            0.000      0.000      0.000   
25%     1958.000         0.416            0.411      0.410      0.000   
50%     1970.000         0.557            0.546      0.550      1.000   
75%     1980.000         0.720            0.712      0.710      1.000   
max     1990.000         1.000            1.000      1.000      1.000   

       lagdemocrat  
count    13588.000  
mean         0.583  
std          0.493  
min          0.000  
25%          0.000  
50%          1.000  
75%          1.000  
max          1.000  
     year  demvoteshare  lagdemvoteshare  dembin  democrat  lagdemocrat
0  1948.0      0.553026         0.469256    0.55       1.0          0.0
1  1948.0

In [7]:
cols = [
    "state",
    "district",
    "year",
    "demvoteshare",
    "lagdemvoteshare",
    "democrat",
    "lagdemocrat"
]

print(df[cols].head(15).to_string(index=False))

          state  district   year  demvoteshare  lagdemvoteshare  democrat  lagdemocrat
01: Connecticut         1 1948.0      0.553026         0.469256       1.0          0.0
01: Connecticut         1 1948.0      0.553026         0.469256       1.0          0.0
01: Connecticut         1 1950.0      0.582441         0.553026       1.0          1.0
01: Connecticut         1 1950.0      0.582441         0.553026       1.0          1.0
01: Connecticut         1 1954.0      0.569626         0.539680       1.0          1.0
01: Connecticut         1 1954.0      0.569626         0.539680       1.0          1.0
01: Connecticut         1 1956.0      0.463041         0.569626       0.0          1.0
01: Connecticut         1 1956.0      0.463041         0.569626       0.0          1.0
01: Connecticut         1 1958.0      0.543411         0.463041       1.0          0.0
01: Connecticut         1 1958.0      0.543411         0.463041       1.0          0.0
01: Connecticut         1 1960.0      0.584

In [9]:
import pandas as pd

# Load the dataset
df = pd.read_stata("../data/lmb-data.dta")

# Check variable labels
reader = pd.io.stata.StataReader("../data/lmb-data.dta")
labels = reader.variable_labels()

variables = [
    "demvoteshare",
    "lagdemvoteshare",
    "democrat",
    "lagdemocrat",
    "dembin"
]

print("Variable labels:")
for name in variables:
    print(name, ":", labels.get(name, "No label"))

# Check duplicate observations
print("\nData checks:")
print("Total rows:", len(df))
print("Total columns:", len(df.columns))
print("Exact duplicate rows:", df.duplicated().sum())

print(
    "Repeated state-district-year combinations:",
    df.duplicated(subset=["state", "district", "year"]).sum()
)

Variable labels:
demvoteshare : 
lagdemvoteshare : 
democrat : 
lagdemocrat : 
dembin : 

Data checks:
Total rows: 13588
Total columns: 178
Exact duplicate rows: 0
Repeated state-district-year combinations: 6446


In [10]:
# Compare two records from the same district and year

subset = df[
    (df["state"].astype(str).str.contains("Connecticut")) &
    (df["district"] == 1) &
    (df["year"] == 1948)
]

print("Number of records:", len(subset))

if len(subset) >= 2:
    row1 = subset.iloc[0]
    row2 = subset.iloc[1]

    differences = pd.DataFrame({
        "Record 1": row1,
        "Record 2": row2
    })

    differences = differences[
        differences["Record 1"].astype(str) !=
        differences["Record 2"].astype(str)
    ]

    print(differences.to_string())

Number of records: 2
               Record 1   Record 2
pooleyear        1949.0     1950.0
aclu_vs             NaN        NaN
acu_vs              NaN        NaN
aca_vs              NaN        NaN
ada_vs              NaN        NaN
afbf_vs             NaN        NaN
afge_vs             NaN        NaN
afscme_vs           NaN        NaN
aft_vs              NaN        NaN
asc_vs              NaN        NaN
bfw_vs              NaN        NaN
bctd_vs             NaN        NaN
carter_vs           NaN        NaN
ccus1_vs            NaN        NaN
ccus2_vs            NaN        NaN
cwla_vs             NaN        NaN
cv_vs               NaN        NaN
cvvf_vs             NaN        NaN
sane_vs             NaN        NaN
cfnfmp_vs           NaN        NaN
aflcio_vs           NaN        NaN
cfsca_vs            NaN        NaN
cfsce_vs            NaN        NaN
cfscd_vs            NaN        NaN
cfscs_vs            NaN        NaN
ccause_vs           NaN        NaN
cw_vs               NaN        NaN

In [11]:
import pandas as pd

subset = df[
    (df["state"].astype(str).str.contains("Connecticut")) &
    (df["district"] == 1) &
    (df["year"] == 1948)
]

a = subset.iloc[0]
b = subset.iloc[1]

# Treat two missing values as equal
different = ~(a.eq(b) | (a.isna() & b.isna()))

comparison = pd.DataFrame({
    "Record 1": a[different],
    "Record 2": b[different]
})

print("Actual differences:", different.sum())
print(comparison.to_string())

Actual differences: 11
            Record 1   Record 2
pooleyear     1949.0     1950.0
adayear         1949       1950
nomada            84         73
realada    64.339996  60.279999
lagnomada       25.0       30.0
avgada          53.0       74.5
n                1.0        2.0
lagged         14.76       9.52
score      64.339996  60.279999
lagscore       14.76       9.52
id               3.0        4.0


## Section 1: Positioning (12 pts)

Two pages maximum in the report. What question the paper asks, what was believed before it, what it changed, and how it has been received, with at least three citations to work that engages with it critically. Then the identification strategy and the assumption it rests on, in your own words.

The prose goes in the report. Keep your notes and the three citations here.

**Critical citations (at least three):**

1. TODO
2. TODO
3. TODO

**The identification strategy and its assumption, in my own words:**

*TODO*

## Section 2: Direct Replication (24 pts)

Reproduce the paper's headline result and report your number beside theirs. You will not match exactly. Document every discrepancy and its likely source: sample construction, variable definition, software defaults, a decision the paper does not describe. Where the paper is ambiguous about a choice, say so: those ambiguities are your Section 3.

### Theirs vs mine

One row per number you replicate. The row label says where the number is in the paper (for example, `Table 2, col 3: effect on log wage`). `gap_pct` is computed for you as 100 × (mine − theirs) / |theirs|. This cell writes `output/headline_table.csv`, which `python reproduce.py` prints and your README quotes.

In [ ]:
headline = pd.DataFrame(
    {
        "paper_estimate": pd.Series(dtype="float"),
        "my_estimate": pd.Series(dtype="float"),
        "gap_pct": pd.Series(dtype="float"),
        "likely_source": pd.Series(dtype="object"),
    },
    index=pd.Index([], name="estimate", dtype="object"),
)

# One line per row; take my_estimate from the cell that computed it, never type it in.
# headline.loc["Table 2, col 3: <what it measures>"] = [<paper's number>, <your variable>, np.nan, "<likely source>"]

headline["gap_pct"] = 100 * (headline["my_estimate"] - headline["paper_estimate"]) / headline["paper_estimate"].abs()
headline.to_csv(ROOT / "output" / "headline_table.csv")
headline

### Discrepancy log

One row per discrepancy, including the ones you could not explain. A clean match reported without investigation is less credible than a 3% gap that is explained.

| # | Estimate (row above) | What differs | Size of the gap | Likely source | How I checked it | Is the paper ambiguous here? |
|---|---|---|---|---|---|---|
| 1 | TODO | TODO | TODO | sample construction / variable definition / software default / undocumented choice | TODO | TODO |
| 2 | | | | | | |

**Ambiguities in the paper, to take into Section 3:**

- TODO